In [4]:
!pip install torch

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as f
from torch.utils.data import DataLoader , TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [30]:
x,y=load_breast_cancer(return_X_y=True)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [31]:
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [32]:
torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.is_available()


True

In [33]:
x_train_scaled = torch.from_numpy(x_train).float()
x_test_scaled= torch.from_numpy(x_test).float()
y_train_scaled = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_scaled= torch.from_numpy(y_test).float().unsqueeze(1)

In [34]:
data=TensorDataset(x_train_scaled,y_train_scaled)
data_loader=DataLoader(data,batch_size=32,shuffle=True)

In [35]:
x_train_scaled.shape

torch.Size([455, 30])

In [146]:
class BCNet(nn.Module):
  def __init__(self):
    super(BCNet,self).__init__()
    self.fc1=nn.Linear(30,64)
    self.fc2=nn.Linear(64,32)
    self.fc3=nn.Linear(32,16)
    self.fc4=nn.Linear(16,8)
    self.fc7=nn.Linear(8,1)
    self.dropout=nn.Dropout(0.4)



  def forward(self,x):
    x=f.relu(self.fc1(x))
    x=f.relu(self.fc2(x))
    x=f.relu(self.fc3(x))
    x=f.relu(self.fc4(x))
    x=self.dropout(x)
    x=f.sigmoid(self.fc7(x))
    return x

In [147]:
model=BCNet()
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [148]:
epochs=10
for epoch in range(epochs):
  model.train()
  runtime_loss=0.0
  for x,y in data_loader:
    optimizer.zero_grad()
    pred=model(x)
    loss=criterion(pred,y)
    loss.backward()
    optimizer.step()
    runtime_loss+=loss.item()
  print(f"Epoch {epoch+1}/{epochs}, Loss: {runtime_loss/len(data_loader)}")

Epoch 1/10, Loss: 0.6803154190381367
Epoch 2/10, Loss: 0.6405720551808675
Epoch 3/10, Loss: 0.584391188621521
Epoch 4/10, Loss: 0.4642301539580027
Epoch 5/10, Loss: 0.33461071451505026
Epoch 6/10, Loss: 0.23601969281832377
Epoch 7/10, Loss: 0.17072461793820062
Epoch 8/10, Loss: 0.13531797234900295
Epoch 9/10, Loss: 0.14204860404133796
Epoch 10/10, Loss: 0.12148921315868695


In [149]:
#Accuracy Check

with torch.no_grad():
  model.eval()
  pred=model(x_test_scaled)
  loss=criterion(pred,y_test_scaled).item()
  print(f"Test Loss: {loss}")
  pred=torch.round(pred)
  correct=torch.sum(pred==y_test_scaled).float().mean().item()
  print(correct/len(y_test_scaled))

Test Loss: 0.050232548266649246
0.9912280701754386
